## Exercise 1

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 10: Transfer Learning
# Niveau: Fortgeschrittene
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import time
import matplotlib

import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("TensorFlow Version:", tf.__version__)

# ── 1. Synthetischen Datensatz erstellen ──────────────────────────────────────
np.random.seed(42)
BILDGROESSE = 48
N_KLASSEN   = 4
N_TRAIN     = 200
N_TEST      = 80

# Klassen: Rot, Grün, Blau, Gelb
farben = [(1,0,0), (0,1,0), (0,0,1), (1,1,0)]

def bilder_erstellen(n, groesse, n_kl):
    X = np.zeros((n, groesse, groesse, 3), dtype="float32")
    y = np.random.randint(0, n_kl, n)
    for i in range(n):
        r, g, b = farben[y[i]]
        X[i, :, :, 0] = r * np.random.uniform(0.6, 1.0)
        X[i, :, :, 1] = g * np.random.uniform(0.6, 1.0)
        X[i, :, :, 2] = b * np.random.uniform(0.6, 1.0)
        X[i] += np.random.rand(groesse, groesse, 3) * 0.2
    return np.clip(X, 0, 1), y

X_train, y_train = bilder_erstellen(N_TRAIN, BILDGROESSE, N_KLASSEN)
X_test,  y_test  = bilder_erstellen(N_TEST,  BILDGROESSE, N_KLASSEN)

print(f"Datensatz: {N_TRAIN} Training, {N_TEST} Test, {N_KLASSEN} Klassen")

# ── 2. VGG16 als Feature-Extractor laden ──────────────────────────────────────
print("\nLade VGG16 (ImageNet-Gewichte)...")
start = time.perf_counter()

# VGG16 benötigt mindestens 32×32 Eingabe
vgg16 = tf.keras.applications.VGG16(
    input_shape=(BILDGROESSE, BILDGROESSE, 3),
    include_top=False,
    weights="imagenet"
)
vgg16.trainable = False  # Alle Schichten einfrieren

ladezeit = time.perf_counter() - start
print(f"VGG16 geladen: {vgg16.count_params():,} Parameter ({ladezeit:.1f}s)")

# Feature-Extractor: Ausgabe nach GlobalAveragePooling
feature_extractor = tf.keras.Sequential([
    vgg16,
    tf.keras.layers.GlobalAveragePooling2D(),
], name="VGG16_Feature_Extractor")

# ── 3. Features extrahieren ───────────────────────────────────────────────────
print("\nExtrahiere Features (Einmaliger Vorwärtsdurchlauf durch VGG16)...")

# VGG16 erwartet vorverarbeitete Eingaben
X_train_prep = tf.keras.applications.vgg16.preprocess_input(X_train * 255.0)
X_test_prep  = tf.keras.applications.vgg16.preprocess_input(X_test  * 255.0)

start = time.perf_counter()
features_train = feature_extractor.predict(X_train_prep, batch_size=16, verbose=1)
features_test  = feature_extractor.predict(X_test_prep,  batch_size=16, verbose=0)
feature_zeit   = time.perf_counter() - start

print(f"Feature-Shape: {features_train.shape} (512 Features pro Bild)")
print(f"Extraktionszeit: {feature_zeit:.1f}s")

# ── 4. Klassifikator auf extrahierten Features trainieren ─────────────────────
# Ansatz 1: Logistische Regression auf Features
print("\nTrainiere Logistische Regression auf extrahierten Features...")
start = time.perf_counter()
lr_klassifikator = LogisticRegression(max_iter=1000, random_state=42)
lr_klassifikator.fit(features_train, y_train)
lr_zeit = time.perf_counter() - start

lr_pred   = lr_klassifikator.predict(features_test)
lr_acc    = accuracy_score(y_test, lr_pred)
print(f"LogReg Genauigkeit: {lr_acc:.4f} ({lr_zeit:.2f}s Trainingszeit)")

# Ansatz 2: Kleines Keras-Netzwerk auf Features
print("\nTrainiere kleines Keras-Netz auf extrahierten Features...")
start = time.perf_counter()
keras_klass = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation="relu", input_shape=(512,)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(N_KLASSEN, activation="softmax"),
], name="Feature_Klassifikator")

keras_klass.compile(optimizer="adam",
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])
history = keras_klass.fit(
    features_train, y_train,
    epochs=30, batch_size=16,
    validation_data=(features_test, y_test), verbose=0
)
keras_zeit = time.perf_counter() - start
keras_acc  = keras_klass.evaluate(features_test, y_test, verbose=0)[1]
print(f"Keras Genauigkeit:  {keras_acc:.4f} ({keras_zeit:.2f}s Trainingszeit)")

# ── 5. Vergleich mit End-to-End Training ─────────────────────────────────────
print("\nVergleich: End-to-End Training (kleines CNN von Grund auf)...")
mini_cnn = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation="relu", padding="same",
                           input_shape=(BILDGROESSE, BILDGROESSE, 3)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), activation="relu", padding="same"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(N_KLASSEN, activation="softmax"),
])
mini_cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])
start = time.perf_counter()
mini_cnn.fit(X_train, y_train, epochs=30, batch_size=16,
             validation_data=(X_test, y_test), verbose=0)
e2e_zeit = time.perf_counter() - start
e2e_acc  = mini_cnn.evaluate(X_test, y_test, verbose=0)[1]
print(f"End-to-End CNN:     {e2e_acc:.4f} ({e2e_zeit:.2f}s Trainingszeit)")

# ── 6. Zusammenfassung ────────────────────────────────────────────────────────
print("\n── Ergebnistabelle ──")
print(f"{'Methode':<30} {'Genauigkeit':>12} {'Trainingszeit':>15}")
print("-" * 58)
print(f"{'VGG16 + LogReg':<30} {lr_acc:>12.4f} {lr_zeit:>14.2f}s")
print(f"{'VGG16 + Keras Dense':<30} {keras_acc:>12.4f} {keras_zeit:>14.2f}s")
print(f"{'CNN from Scratch':<30} {e2e_acc:>12.4f} {e2e_zeit:>14.2f}s")

# ── 7. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history["val_accuracy"], label="VGG16 + Keras", linewidth=2)
axes[0].set_title("Validierungs-Genauigkeit (VGG16 Features)")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[0].grid(True)

methoden = ["VGG16+\nLogReg", "VGG16+\nKeras", "CNN\nScratch"]
accs     = [lr_acc, keras_acc, e2e_acc]
farben_b = ["#3498db", "#2ecc71", "#e74c3c"]
axes[1].bar(methoden, accs, color=farben_b, edgecolor="black")
axes[1].set_title("Test-Genauigkeit Vergleich")
axes[1].set_ylabel("Genauigkeit")
axes[1].set_ylim(0, 1.1)
for i, a in enumerate(accs):
    axes[1].text(i, a + 0.02, f"{a:.3f}", ha="center", fontsize=11)
axes[1].grid(True, axis="y", alpha=0.3)

plt.suptitle("VGG16 Feature Extractor vs. Training von Grund auf", fontsize=13)
plt.tight_layout()
plt.savefig("F10_1_vgg16_features.png", dpi=100)
plt.show()
print("Diagramm gespeichert: F10_1_vgg16_features.png")


## Exercise 2

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 10: Transfer Learning
# Niveau: Fortgeschrittene
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Zwei Domänen erstellen (Quell- und Ziel-Domäne) ───────────────────────
np.random.seed(42)
BILDGROESSE = 48
N_KLASSEN   = 2
N_PROBEN    = 300

def bilder_erstellen(n, groesse, n_kl, helligkeits_offset=0.0, varianz=0.2):
    """
    Erstellt farbige Bilder.
    helligkeits_offset: positiv = helle Bilder (Quelldomain),
                         negativ = dunkle Bilder (Zieldomäne)
    """
    X = np.zeros((n, groesse, groesse, 3), dtype="float32")
    y = np.random.randint(0, n_kl, n)
    for i in range(n):
        if y[i] == 0:
            X[i, :, :, 0] = 0.8   # Rot
        else:
            X[i, :, :, 2] = 0.8   # Blau
        X[i] += np.random.rand(groesse, groesse, 3) * varianz
        X[i] += helligkeits_offset
    return np.clip(X, 0, 1), y

# Quell-Domäne: helle Bilder (Originalverteilung)
X_quelle, y_quelle = bilder_erstellen(N_PROBEN, BILDGROESSE, N_KLASSEN,
                                       helligkeits_offset=0.3)
# Ziel-Domäne: dunkle Bilder (Domain Shift)
X_ziel,   y_ziel   = bilder_erstellen(N_PROBEN, BILDGROESSE, N_KLASSEN,
                                       helligkeits_offset=-0.2)

# Split
split = int(0.8 * N_PROBEN)
X_q_tr, X_q_te = X_quelle[:split], X_quelle[split:]
X_z_tr, X_z_te = X_ziel[:split],   X_ziel[split:]
y_q_tr, y_q_te = y_quelle[:split], y_quelle[split:]
y_z_tr, y_z_te = y_ziel[:split],   y_ziel[split:]

print("Quell-Domäne: helle Bilder (offset=+0.3)")
print("Ziel-Domäne:  dunkle Bilder (offset=-0.2)")
print(f"Quelle: μ_hellig={X_quelle.mean():.3f}  |  Ziel: μ_hellig={X_ziel.mean():.3f}")

# ── 2. Basismodell auf Quell-Domäne trainieren ────────────────────────────────
def cnn_erstellen(n_kl=N_KLASSEN, groesse=BILDGROESSE):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(groesse, groesse, 3),
        include_top=False, weights="imagenet"
    )
    base.trainable = False
    m = tf.keras.Sequential([
        base,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(n_kl, activation="softmax"),
    ])
    m.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

print("\nTrainiere Modell auf Quell-Domäne...")
modell = cnn_erstellen()
modell.fit(X_q_tr, y_q_tr, epochs=5, batch_size=16,
           validation_split=0.1, verbose=1)

# ── 3. Leistungsabfall in Ziel-Domäne zeigen ─────────────────────────────────
acc_quelle = modell.evaluate(X_q_te, y_q_te, verbose=0)[1]
acc_ziel_vor = modell.evaluate(X_z_te, y_z_te, verbose=0)[1]

print(f"\nGenauigkeit auf Quell-Domäne:     {acc_quelle:.4f}")
print(f"Genauigkeit auf Ziel-Domäne:      {acc_ziel_vor:.4f}  ← Domain Shift!")
print(f"Leistungsabfall:                  {(acc_quelle - acc_ziel_vor)*100:.1f}%")

# ── 4. Domain Adaptation: Feature-Normalisierung ─────────────────────────────
print("\n── Domain Adaptation: Feature-Normalisierung ──")

# Methode 1: Histogramm-Angleichung (einfach: Mittlewert/Std angleichen)
q_mean, q_std = X_quelle.mean(), X_quelle.std()
z_mean, z_std = X_ziel.mean(),   X_ziel.std()

# Zielbilder auf Quell-Statistik normalisieren
X_z_adaptiert = (X_ziel - z_mean) / (z_std + 1e-8) * q_std + q_mean
X_z_adaptiert = np.clip(X_z_adaptiert, 0, 1).astype("float32")

X_z_te_adapt = X_z_adaptiert[split:]
acc_ziel_adapt = modell.evaluate(X_z_te_adapt, y_z_te, verbose=0)[1]

print(f"Genauigkeit nach Adaptation:      {acc_ziel_adapt:.4f}")
print(f"Verbesserung durch Adaptation:    {(acc_ziel_adapt - acc_ziel_vor)*100:+.1f}%")

# Methode 2: Minimales Fine-Tuning auf wenigen Ziel-Domänen-Beispielen
print("\n── Methode 2: Few-Shot Fine-Tuning auf Ziel-Domäne ──")
modell_ft = cnn_erstellen()
modell_ft.fit(X_q_tr, y_q_tr, epochs=5, batch_size=16,
              validation_split=0.1, verbose=0)

# Letzte Schicht mit 50 Ziel-Domänen-Beispielen feinabstimmen
modell_ft.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
modell_ft.fit(X_z_tr[:50], y_z_tr[:50], epochs=5, batch_size=8, verbose=0)
acc_ziel_ft = modell_ft.evaluate(X_z_te, y_z_te, verbose=0)[1]
print(f"Genauigkeit nach Fine-Tuning:     {acc_ziel_ft:.4f}")

# ── 5. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Beispielbilder aus beiden Domänen
axes[0].imshow(X_quelle[0])
axes[0].set_title("Quell-Domäne\n(hell)")
axes[0].axis("off")

ax1_ins = axes[0].inset_axes([0.5, 0, 0.5, 1])
ax1_ins.imshow(X_ziel[0])
ax1_ins.set_title("Ziel-Dom.\n(dunkel)", fontsize=8)
ax1_ins.axis("off")

# Balkenvergleich
methoden = ["Quell-\nDomäne", "Ziel-Dom.\n(ohne Adapt.)",
            "Ziel-Dom.\n(Norm. Adapt.)", "Ziel-Dom.\n(Fine-Tuning)"]
accs     = [acc_quelle, acc_ziel_vor, acc_ziel_adapt, acc_ziel_ft]
farben_b = ["#2ecc71", "#e74c3c", "#f39c12", "#3498db"]
axes[1].bar(methoden, accs, color=farben_b, edgecolor="black")
axes[1].set_title("Domain Adaptation: Genauigkeitsvergleich")
axes[1].set_ylabel("Genauigkeit")
axes[1].set_ylim(0, 1.15)
for i, a in enumerate(accs):
    axes[1].text(i, a + 0.02, f"{a:.3f}", ha="center", fontsize=10)
axes[1].grid(True, axis="y", alpha=0.3)

# Verteilung der Helligkeitswerte
axes[2].hist(X_quelle.flatten(), bins=50, alpha=0.6, color="#2ecc71",
             label="Quell-Dom.", density=True)
axes[2].hist(X_ziel.flatten(), bins=50, alpha=0.6, color="#e74c3c",
             label="Ziel-Dom.", density=True)
axes[2].hist(X_z_adaptiert.flatten(), bins=50, alpha=0.4, color="#f39c12",
             label="Ziel (adaptiert)", density=True, linestyle="--")
axes[2].set_title("Helligkeitsverteilung beider Domänen")
axes[2].set_xlabel("Pixelwert")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("Domain Adaptation: Quell- vs. Ziel-Domäne (Helligkeits-Shift)",
             fontsize=12)
plt.tight_layout()
plt.savefig("F10_2_domain_adaptation.png", dpi=100)
plt.show()
print("Diagramm gespeichert: F10_2_domain_adaptation.png")


## Exercise 3

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 10: Transfer Learning
# Niveau: Fortgeschrittene
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Synthetische Bilder erstellen (Form + Farbe) ───────────────────────────
from PIL import Image, ImageDraw
np.random.seed(42)

BILDGROESSE    = 48
N_FORMEN       = 3   # Kreis, Quadrat, Dreieck
N_FARBEN       = 3   # Rot, Grün, Blau
N_PROBEN       = 500

def bild_erstellen(form_id, farb_id, groesse=BILDGROESSE):
    """
    form_id: 0=Kreis, 1=Quadrat, 2=Dreieck
    farb_id: 0=Rot, 1=Grün, 2=Blau
    """
    img  = Image.new("RGB", (groesse, groesse), color=(180, 180, 180))
    draw = ImageDraw.Draw(img)
    m = groesse // 2
    r = groesse // 4
    rgb_map = {0: (220, 50, 50), 1: (50, 200, 50), 2: (50, 50, 220)}
    farbe = rgb_map[farb_id]

    if form_id == 0:   # Kreis
        draw.ellipse([m-r, m-r, m+r, m+r], fill=farbe)
    elif form_id == 1:  # Quadrat
        draw.rectangle([m-r, m-r, m+r, m+r], fill=farbe)
    elif form_id == 2:  # Dreieck
        draw.polygon([(m, m-r), (m-r, m+r), (m+r, m+r)], fill=farbe)

    arr = np.array(img).astype("float32") / 255.0
    arr += np.random.randn(groesse, groesse, 3) * 0.05
    return np.clip(arr, 0, 1)

# Datensatz generieren
X_list, y_form_list, y_farbe_list = [], [], []
for _ in range(N_PROBEN):
    f_id = np.random.randint(N_FORMEN)
    c_id = np.random.randint(N_FARBEN)
    X_list.append(bild_erstellen(f_id, c_id))
    y_form_list.append(f_id)
    y_farbe_list.append(c_id)

X = np.array(X_list, dtype="float32")
y_form  = np.array(y_form_list,  dtype="int32")
y_farbe = np.array(y_farbe_list, dtype="int32")

# Mischen
idx = np.random.permutation(N_PROBEN)
X, y_form, y_farbe = X[idx], y_form[idx], y_farbe[idx]

split = int(0.8 * N_PROBEN)
X_tr, X_te = X[:split], X[split:]
y_form_tr,  y_form_te  = y_form[:split],  y_form[split:]
y_farbe_tr, y_farbe_te = y_farbe[:split], y_farbe[split:]

print(f"Datensatz: {X.shape}, Klassen: Form ({N_FORMEN}), Farbe ({N_FARBEN})")

# ── 2. Multi-Task-Modell aufbauen (ein Backbone, zwei Ausgabe-Köpfe) ──────────
# Gemeinsame Basis (Shared Layers)
eingabe = tf.keras.Input(shape=(BILDGROESSE, BILDGROESSE, 3), name="bild_eingabe")

# Geteilter Feature-Extraktor
x = tf.keras.layers.Conv2D(32, (3,3), activation="relu", padding="same")(eingabe)
x = tf.keras.layers.MaxPooling2D((2,2))(x)
x = tf.keras.layers.Conv2D(64, (3,3), activation="relu", padding="same")(x)
x = tf.keras.layers.MaxPooling2D((2,2))(x)
x = tf.keras.layers.Conv2D(128, (3,3), activation="relu", padding="same")(x)
gemeinsame_features = tf.keras.layers.GlobalAveragePooling2D(name="backbone_out")(x)

# ── Kopf 1: Form-Klassifikation ──
kopf_form = tf.keras.layers.Dense(64, activation="relu", name="form_dense")(gemeinsame_features)
ausgabe_form = tf.keras.layers.Dense(
    N_FORMEN, activation="softmax", name="form_ausgabe"
)(kopf_form)

# ── Kopf 2: Farb-Klassifikation ──
kopf_farbe = tf.keras.layers.Dense(64, activation="relu", name="farbe_dense")(gemeinsame_features)
ausgabe_farbe = tf.keras.layers.Dense(
    N_FARBEN, activation="softmax", name="farbe_ausgabe"
)(kopf_farbe)

# Gemeinsames Modell mit 2 Ausgaben
modell = tf.keras.Model(
    inputs=eingabe,
    outputs=[ausgabe_form, ausgabe_farbe],
    name="MultiTask_Form_Farbe"
)

# Kombinierter Verlust: Gewichtetes Mittel beider Aufgaben
modell.compile(
    optimizer="adam",
    loss={
        "form_ausgabe":  "sparse_categorical_crossentropy",
        "farbe_ausgabe": "sparse_categorical_crossentropy",
    },
    loss_weights={
        "form_ausgabe":  1.0,   # Beide Aufgaben gleich gewichtet
        "farbe_ausgabe": 1.0,
    },
    metrics={
        "form_ausgabe":  "accuracy",
        "farbe_ausgabe": "accuracy",
    }
)
modell.summary()

# ── 3. Training ───────────────────────────────────────────────────────────────
print("\nTrainiere Multi-Task-Modell...")
history = modell.fit(
    X_tr,
    {"form_ausgabe": y_form_tr, "farbe_ausgabe": y_farbe_tr},
    epochs=20,
    batch_size=32,
    validation_data=(
        X_te,
        {"form_ausgabe": y_form_te, "farbe_ausgabe": y_farbe_te}
    ),
    verbose=1
)

# ── 4. Evaluation ─────────────────────────────────────────────────────────────
ergebnisse = modell.evaluate(
    X_te,
    {"form_ausgabe": y_form_te, "farbe_ausgabe": y_farbe_te},
    verbose=0
)
print(f"\nForm-Genauigkeit:  {ergebnisse[-2]:.4f}")
print(f"Farbe-Genauigkeit: {ergebnisse[-1]:.4f}")

# ── 5. Vorhersagen ────────────────────────────────────────────────────────────
pred_form, pred_farbe = modell.predict(X_te[:6], verbose=0)
formen_namen = ["Kreis", "Quadrat", "Dreieck"]
farben_namen = ["Rot",   "Grün",    "Blau"]

print("\nBeispiel-Vorhersagen:")
for i in range(6):
    pf = np.argmax(pred_form[i])
    pc = np.argmax(pred_farbe[i])
    print(f"  {i+1}: Form={formen_namen[pf]}({y_form_te[i]}) "
          f"Farbe={farben_namen[pc]}({y_farbe_te[i]})")

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for key, farbe, label in [
    ("form_ausgabe_accuracy",  "#3498db", "Form-Acc"),
    ("farbe_ausgabe_accuracy", "#e74c3c", "Farbe-Acc"),
]:
    val_key = f"val_{key}"
    if key in history.history:
        axes[0].plot(history.history[val_key], color=farbe,
                     label=f"Val {label}", linewidth=2)

axes[0].set_title("Validierungs-Genauigkeit (beide Aufgaben)")
axes[0].set_xlabel("Epoche")
axes[0].set_ylabel("Genauigkeit")
axes[0].legend()
axes[0].grid(True)

# Beispielbilder mit Vorhersagen
fig2, ax2s = plt.subplots(1, 6, figsize=(14, 3))
for i in range(6):
    pf = np.argmax(pred_form[i])
    pc = np.argmax(pred_farbe[i])
    ax2s[i].imshow(X_te[i])
    korrekt = (pf == y_form_te[i]) and (pc == y_farbe_te[i])
    ax2s[i].set_title(f"F:{formen_namen[pf]}\nC:{farben_namen[pc]}",
                       color="green" if korrekt else "red", fontsize=8)
    ax2s[i].axis("off")
plt.suptitle("Multi-Task Vorhersagen (Form + Farbe)", fontsize=12)
plt.tight_layout()
plt.savefig("F10_3_multi_task_beispiele.png", dpi=100)
plt.show()

# Gesamt-Verlauf
axes[1].plot(history.history.get("loss", []), label="Gesamt-Loss (Train)")
axes[1].plot(history.history.get("val_loss", []), label="Gesamt-Loss (Val)")
axes[1].set_title("Gesamt-Verlust (kombiniert)")
axes[1].set_xlabel("Epoche")
axes[1].legend()
axes[1].grid(True)
fig.tight_layout()
fig.savefig("F10_3_multi_task_training.png", dpi=100)
plt.show()
print("Diagramme gespeichert: F10_3_*.png")
